In [2]:
import numpy as np
import pandas as pd
import os, re, math, platform
from pathlib import Path
import matplotlib.pyplot as plt
import json
import joblib
from scipy.stats import randint as sp_randint
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import RandomizedSearchCV
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import matthews_corrcoef, confusion_matrix
from sklearn.metrics import precision_recall_curve, roc_curve, auc, fbeta_score
from imblearn.metrics import geometric_mean_score
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier 
from xgboost import plot_importance
from sklearn.ensemble import GradientBoostingClassifier,RandomForestClassifier,ExtraTreesClassifier,AdaBoostClassifier
from sklearn.linear_model import SGDClassifier
#!pip install lightgbm
from sklearn.ensemble import BaggingClassifier
#from sklearn.linear_model import ElasticNet
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC
from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis as PA
from modlamp.descriptors import PeptideDescriptor, GlobalDescriptor
from matplotlib import pyplot
from sklearn.metrics import matthews_corrcoef, confusion_matrix,precision_recall_curve, roc_curve, auc, fbeta_score,roc_auc_score
from fea_extract import read_fasta,insert_AAC,insert_DPC,insert_CKSAAGP,insert_CTD,insert_PAAC,insert_AAI,insert_GTPC,insert_QSO,insert_AAE,insert_PSAAC,insert_word2int,insert_ASDC
import warnings 
from collections import Counter
from tools import cv,evaluate
import matplotlib.pyplot as plt
import seaborn as sns
Path('./results/balance_eval_AUC/').mkdir(exist_ok=True,parents=True)
warnings.filterwarnings('ignore')
seed=10

In [10]:
def pro_data(seq):
    df_n = insert_PAAC(seq)
    df_n = insert_AAC(df_n)
    df_n = insert_CKSAAGP(df_n)
    df_n = insert_CTD(df_n)
    df_n = insert_DPC(df_n)
    df_n = insert_GTPC(df_n)
    df_n = insert_QSO(df_n)
    df_n = insert_AAE(df_n)
    df_n = insert_ASDC(df_n)
    #df_n = insert_word2int(df_n)
    return df_n

In [11]:
seq_X_train = pd.read_csv('data/train/X_train.csv')
#seq_X_test = pd.read_csv('data/test/X_test.csv')
seq_y_train = pd.read_csv('data/train/y_train.csv')
#seq_y_test = pd.read_csv('data/test/y_test.csv')

In [12]:
Seq_X_train = pro_data(seq_X_train)
#Seq_X_test = pro_data(seq_X_test)

In [13]:
Seq_X_train.to_csv('data/train/Seq_X_train_all.csv',index=False)
#Seq_X_test.to_csv('data/test/Seq_X_test_all.csv',index=False)

In [14]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier
from mlxtend.classifier import StackingClassifier

from sklearn.svm import LinearSVC

ml = ["DT","LGBM","XGBoost","RF","ET","GBDT","SVM","MLP","LR","AB","voting_clf","stacking","Two_eclf"]

layer_one= [  LGBMClassifier(random_state = seed),ExtraTreesClassifier(random_state=seed),RandomForestClassifier(random_state=seed),MLPClassifier(hidden_layer_sizes=700,learning_rate="adaptive",random_state=seed)]
                     #('rf5',GaussianNB()),
#                   ('SVM', SVC(kernel='linear',probability=True))]
layer_two = [SVC(random_state=seed,probability=True), ExtraTreesClassifier(random_state=seed),RandomForestClassifier(random_state=seed),LGBMClassifier(random_state = seed)]
             #GaussianNB()]
#                   ('SVC', SVC(kernel='linear',probability=True))]

layer_two_meta= StackingClassifier(classifiers = layer_two, meta_classifier=LogisticRegression())    
DT  = DecisionTreeClassifier(random_state=seed)
LGBM = LGBMClassifier(random_state = seed)
GBDT = GradientBoostingClassifier(random_state=seed)
ET = ExtraTreesClassifier(random_state=seed)
SVM = SVC(random_state=seed,probability=True)
MLP = MLPClassifier(hidden_layer_sizes=700,learning_rate="adaptive",random_state=seed)
RF = RandomForestClassifier(random_state=seed)
XGBoost = XGBClassifier(random_state=seed)
LR  = LogisticRegression(random_state=seed,solver='liblinear')
AB = AdaBoostClassifier(random_state=seed)
#SDG_Classifier = SGDClassifier(max_iter=1000, tol=1e-3) 
voting_clf = VotingClassifier(estimators = [('lgbm',LGBM),('rf',RF),('xgb',XGBoost),('ET',ET),('GBDT',GBDT)], voting = 'soft')
Bagging_clf = BaggingClassifier(base_estimator=RandomForestClassifier(random_state=seed))
#VOT_STACK = StackingClassifier(classifiers = voting_clf, meta_classifier=LR)
stacking = StackingClassifier(classifiers=[LGBM,RF,XGBoost,ET, GBDT],meta_classifier=LR)
Two_eclf = StackingClassifier(classifiers=layer_one, meta_classifier=layer_two_meta)


In [16]:
fea = Seq_X_train.columns[2:]
X_train = Seq_X_train[fea].to_numpy()
#X_test = Seq_X_test[fea].to_numpy()
y_train = seq_y_train.to_numpy()
#y_test = seq_y_test.to_numpy()

In [17]:
len(fea)

1296

In [18]:
index = []
ALL_eval=pd.DataFrame()
for i in ml:
    print('process_{}'.format(i))
    model = eval(i)
    Evals = cv(model,X_train,y_train)
    ALL_eval = pd.concat([ALL_eval,Evals],axis=1)
    index.append("{}".format(i))
ALL_eval.columns = index

process_DT
process_LGBM
[LightGBM] [Info] Number of positive: 857, number of negative: 615
[LightGBM] [Warning] Auto-choosing col-wise multi-threading, the overhead of testing was 0.042721 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 66055
[LightGBM] [Info] Number of data points in the train set: 1472, number of used features: 1132
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.582201 -> initscore=0.331816
[LightGBM] [Info] Start training from score 0.331816
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 871, number of negative: 601
[LightGBM] [Warning] Auto-choosing col-wise multi-threading, the overhead of testing was 0.042090 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 65213
[LightGBM] [Info] Number of data points in the train set: 1472, nu

In [19]:
ALL_eval
#for comparision to physiochemcial based method for attribution to GAN based data augmentation

,DT,LGBM,XGBoost,RF,ET,GBDT,SVM,MLP,LR,AB,voting_clf,stacking,Two_eclf
ACC,0.892391,0.946739,0.941304,0.942935,0.945109,0.951087,0.930978,0.936413,0.929891,0.928261,0.948913,0.948913,0.947283
F1,0.908218,0.953813,0.949372,0.950239,0.952333,0.957570,0.941259,0.944954,0.939433,0.938071,0.955711,0.955869,0.954597
F2,0.912538,0.949623,0.949010,0.943121,0.947894,0.954992,0.949413,0.945713,0.939228,0.938619,0.952126,0.952796,0.953332
GMean,0.888088,0.946917,0.940276,0.944026,0.945351,0.950890,0.925809,0.934255,0.927969,0.926061,0.949141,0.948792,0.946297
SEN,0.915652,0.946865,0.948819,0.938461,0.944974,0.953315,0.954962,0.946233,0.939123,0.938993,0.949796,0.950796,0.952510
PREC,0.901904,0.960937,0.950173,0.962457,0.959890,0.962037,0.928096,0.943743,0.939901,0.937193,0.961929,0.961195,0.956792
SPEC,0.862038,0.947006,0.931966,0.949708,0.945760,0.948570,0.897678,0.922505,0.917037,0.913339,0.948613,0.946903,0.940182
MCC,0.779495,0.890971,0.879728,0.883760,0.887684,0.899921,0.857988,0.869365,0.856028,0.852475,0.895634,0.895409,0.891669
AUC,0.888845,0.990021,0.989186,0.988780,0.988298,0.990505,0.981295,0.988429,0.982768,0.979112,0.990765,0.967149,0.957496
AUPR,0.874977,0.993030,0.992347,0.992081,0.991800,0.993341,0.985364,0.991684,0.987932,0.981836,0.993576,0.965285,0.953724
